In [ ]:
import tz_pypsa
from tz_pypsa.model import Model

from tz_pypsa.constraints import (constr_bus_self_sufficiency, 
                                  constr_cumulative_p_nom, 
                                  constr_policy_targets, 
                                  constr_max_annual_utilisation)

# from tz_pypsa.constraints import constr_cofiring_ccs_generation_join_plant

# Load a stock model with user-defined configuration
network = (
    Model.load_model(
        'ASEAN', 
        frequency = '24h',
        select_nodes = ['MYSPE',
                        'MYSSK',
                        'MYSSH',
                        'SGPXX',
                        'THASO',
                        'THACE',
                        'THANO',
                        # 'LAOXX',
                        # 'VNMCE',
                        # 'VNMSO',
                        # 'VNMNO',
                        # 'KHMXX',
                        # 'MMRXX',
                        'IDNKA'],
        years = [2030],
        backstop = True,
        set_global_constraints = False,
    )
)                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                           

In [ ]:
network.generators.p_nom_extendable = True

In [ ]:
# network.generators.p_nom_extendable.loc[network.generators.index.str.contains('cofiring')] = False

In [ ]:
network.links.p_nom_extendable = False

In [ ]:
network.storage_units.p_nom_extendable.loc[network.storage_units.index.str.contains('lithium')] = True
network.storage_units.p_nom_extendable.loc[network.storage_units.index.str.contains('pumped')] = False

In [ ]:
network.optimize.create_model()

In [ ]:
# constr_cofiring_ccs_generation_join_plant(network,
#                                           clean_generator=['MYSPE-coal-biomass-cofiring-clean-ext-2030'],
#                                           fossil_generator=['MYSPE-coal-biomass-cofiring-fossil-ext-2030'],
#                                           model_frequency = 12)

In [ ]:
# # solve the model
# network.optimize(
#   solver_name='highs',
# )

In [ ]:
# network.export_to_netcdf('/home/yantidiah/tza-pypsa/output/calibration/apg_cferelated-1h_24Feb_12PM.nc')

In [ ]:
# # Instantiate the linopy model
# lp_model = network.optimize.create_model()

# add stock constraints ((optional))
constr_bus_self_sufficiency(network, 
                            # lp_model,
                            min_self_sufficiency = 0.6)

# add technical potential constraints across all years 
# (cannot be activatee for calibration - single year)
# constr_cumulative_p_nom(network, 
#                         # lp_model
#                         )

# Add policy targets as custom linopy constraints - still need to be merged from APG_v2 branch
constr_policy_targets(network,
                    #   lp_model,
                      'ASEAN') # This requires that a CSV with the policy targets are provided within the stock model data

# Add constraint on the maximum utilisation rate of coal plants
constr_max_annual_utilisation(network, 
                            #   lp_model, 
                              carriers = ['coal','gas','biomass','geothermal','bioenergy','oil'], 
                              max_utilisation_rate = 0.85)

# # Let's solve the model again
# network.optimize.solve_model()

In [ ]:
network.optimize.solve_model()

# CHECKING RESULTS

In [ ]:
network.model.constraints

In [ ]:
network.model.constraints['min_gen_by_MYSPE']

In [ ]:
network.loads_t.p_set.sum() / 1e6

In [ ]:
network.statistics(groupby=['bus','carrier']).loc['Generator']

In [ ]:
network.statistics.expanded_capacity()

In [ ]:
network.generators.p_nom_opt.filter(regex='photo')

In [ ]:
# --- CHECKING BACKSTOP --- #

network.generators_t.p.filter(like='Backstop').sum()

In [ ]:
# --- CHECKING ENERGY BALANCE --- #

import tz_pypsa
fig = tz_pypsa.plotting.energy_balance(network,2030)
fig.update_layout(width=1000,height=500)
fig.show()

In [ ]:
# --- CHECKING INSTALLED CAPACITY --- #

fig = tz_pypsa.plotting.total_capacity(network,2030)
fig.update_layout(width=1000,height=500)
fig.show()

In [ ]:
tz_pypsa.plotting.dispatch(network,2030,'MYS')

In [ ]:
# --- CHECKING GENERATION PER TYPE --- #

import plotly.express as px

loads = (
    network
    .loads_t
    .p_set
    .resample('YE')
    .sum()
    .div(1e6)
    .melt()
    .sort_values(by='Load')
    .reset_index(drop=True)
)

total_generation = (
    network
    .generators_t
    .p
    # .loc[2023]
    .resample('YE')
    .sum()
    .groupby([network.generators.bus, network.generators.carrier], axis=1)
    .sum()
    .div(1e6)
    .melt()
    .sort_values(by='bus')
)

# define order for x-axis
cat_order = total_generation.sort_values(by='bus').bus.unique().tolist()


fig = px.bar(
    total_generation, 
    x="bus", 
    y="value", 
    color = "carrier",
    category_orders={'bus' : cat_order},
    #barmode="group",
)

fig.add_scatter(
    x=loads.Load,
    y=loads.value,
    mode='markers',
    name='Load',
    marker=dict(
        size=7,
        color='red',
        symbol='circle',
    ),
)

fig.update_layout(
        yaxis_title= 'Generation (TWh)',
        xaxis_title='',
        # title='Installed capacity in ' + str(2023),
        width=1000,
        height=500,
        xaxis_tickangle=-45
    )

fig

In [ ]:
# --- CALIBRATION 1a: GENERATION 2023- Official data vs model (Only MYS and SGP) --- #

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Extract model results for Singapore and Malaysia
sgp_gen_pyspsa = (
    network
    .generators_t
    .p
    # .loc[2023]
    .filter(like="SGP")
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().sum().div(1e6)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'gen_pypsa'})
)
mys_gen_pyspsa = (
    network
    .generators_t
    .p
    # .loc[2023]
    .filter(like=("MYS"))
    .groupby(by=[network.generators.carrier],axis=1)
    .sum().sum().div(1e6)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'gen_pypsa'})
)

# Extract official data for Singapore and Malaysia
gen_ofc_doc = pd.read_excel('/home/yantidiah/tza-pypsa/Generation_2023_official.xlsx')
sgp_gen_ofc = (
    gen_ofc_doc[gen_ofc_doc['Country_code'] == 'SGP']
    .reset_index().rename(columns={'Generation_GWh':'gen_ofc'})
)
mys_gen_ofc = (
    gen_ofc_doc[gen_ofc_doc['Country_code'] == 'MYS']
    .reset_index().rename(columns={'Generation_GWh':'gen_ofc'})
)

# Combine both
sgp_gen = sgp_gen_pyspsa.merge(sgp_gen_ofc,how='outer',on='Fuel')
mys_gen = mys_gen_pyspsa.merge(mys_gen_ofc,how='outer',on='Fuel')

# # subplot for SGP and MYS
fig = make_subplots(rows=2, cols=1,shared_xaxes=False, subplot_titles=('Singapore','Malaysia'))

fig.add_trace(go.Bar(x=sgp_gen['Fuel'], y=sgp_gen['gen_ofc'].div(1e3),name='SGP ref'),1,1)
fig.add_trace(go.Bar(x=sgp_gen['Fuel'], y=sgp_gen['gen_pypsa'],name='SGP results'),1,1)

fig.add_trace(go.Bar(x=mys_gen['Fuel'], y=mys_gen['gen_ofc'].div(1e3),name='MYS ref'),2,1)
fig.add_trace(go.Bar(x=mys_gen['Fuel'], y=mys_gen['gen_pypsa'],name='MYS results'),2,1)

fig.update_layout(title_text="2023 GENERATION (TWh) COMPARISON", height=500)

fig.show()

In [ ]:
# --- CALIBRATION 1b: GENERATION 2023- Official data vs model (all Indochina) --- #

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Extract country data and results
country_list = ['KHM','LAO','MMR','MYS','SGP','THA','VNM']
fig = make_subplots(rows=len(country_list), cols=1,shared_xaxes=False, subplot_titles=country_list)
for country in country_list:

    # Pypsa results
    gen_pyspsa = (
        network
        .generators_t
        .p
        # .loc[2023]
        .filter(like=country)
        .groupby(by=[network.generators.carrier],axis=1)
        .sum().sum().div(1e6)
        .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'gen_pypsa'})
    )

    # Extract official data for Singapore and Malaysia
    gen_ofc_doc = pd.read_excel('/home/yantidiah/tza-pypsa/Generation_2023_official.xlsx')
    gen_ofc = (
        gen_ofc_doc[gen_ofc_doc['Country_code'] == country]
        .reset_index().rename(columns={'Generation_GWh':'gen_ofc'})
    )
    # Combine both
    gen = gen_pyspsa.merge(gen_ofc,how='outer',on='Fuel')

    # subplot
    fig.add_trace(go.Bar(x=gen['Fuel'], y=gen['gen_ofc'].div(1e3),name=country+' ref'),country_list.index(country)+1,1)
    fig.add_trace(go.Bar(x=gen['Fuel'], y=gen['gen_pypsa'],name=country+' results'),country_list.index(country)+1,1)

fig.update_layout(title_text="2023 GENERATION (TWh) COMPARISON",
                  width=1000,height=2000)

fig.show()

In [ ]:
# --- CALIBRATION 2: CAPACITY FACTOR 2023 - Official data vs model (only MYS and SGP) --- #

# Extract CF results for Singapore and Malaysia
cf_pypsa = (network.statistics.capacity_factor(groupby=['bus','carrier'])
            .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cf_pypsa'})
)
cf_pypsa['Fuel'] = cf_pypsa['Fuel'].str.lower()

# cf_pypsa['Generator'].loc[['SGPXX','MYSPE']]

# Official doc results
cap_sgp = (
    network.generators.p_nom
    .filter(like='SGP')
    .groupby(by=[network.generators.carrier])
    .sum().div(1e3)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cap'})
)
sgp_cf = sgp_gen.merge(cap_sgp,how='outer',on='Fuel')
sgp_cf['cf_ofc'] = sgp_cf['gen_ofc'].div(sgp_cf['p_nom'] * 8760)
sgp_cf = sgp_cf.merge(cf_pypsa.loc[(cf_pypsa['bus'] == 'SGPXX') & (cf_pypsa['component'] == 'Generator')],
                      how='outer',on='Fuel')

cap_mys = (
    network.generators.p_nom
    .filter(like='MYS')
    .groupby(by=[network.generators.carrier])
    .sum().div(1e3)
    .to_frame().reset_index().rename(columns={'carrier':'Fuel',0:'cap'})
)
mys_cf = mys_gen.merge(cap_mys,how='outer',on='Fuel')
mys_cf['cf_ofc'] = mys_cf['gen_ofc'].div(mys_cf['p_nom'] * 8760)
mys_cf = mys_cf.merge(cf_pypsa.loc[(cf_pypsa['bus'] == 'MYSPE') & (cf_pypsa['component'] == 'Generator')],
                      how='outer',on='Fuel')

# subplot for SGP and MYS
fig = go.Figure()
fig = make_subplots(rows=2, cols=1,shared_xaxes=False, subplot_titles=('Singapore','Malaysia'))

fig.add_trace(go.Bar(x=sgp_cf['Fuel'], y=sgp_cf['cf_ofc'],name='SGP ref'),1,1)
fig.add_trace(go.Bar(x=sgp_cf['Fuel'], y=sgp_cf['cf_pypsa'],name='SGP results'),1,1)

fig.add_trace(go.Bar(x=mys_cf['Fuel'], y=mys_cf['cf_ofc'],name='MYS ref'),2,1)
fig.add_trace(go.Bar(x=mys_cf['Fuel'], y=mys_cf['cf_pypsa'],name='MYS results'),2,1)

fig.update_layout(title_text="2023 CF COMPARISON", height=500)

fig.show()

In [ ]:
# --- CALIBRATION 3a: DISPATCH MONTHLY 2023 - Singapore (model run) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='SGP',resample='D', show_exports=True, show_imports=True)
fig.update_layout(title_text='SGP dispatch 2023 (model run)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3b: DISPATCH MONTHLY 2023 - Singapore (ofc) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

sgp_dispatch_ofc = pd.read_csv('/home/yantidiah/tza-pypsa/SGP_hourly_demand_solargen_cost_2023-2024.csv')
sgp_dispatch_ofc['timestamp'] = pd.to_datetime(sgp_dispatch_ofc['timestamp'])
sgp_dispatch_ofc = sgp_dispatch_ofc[sgp_dispatch_ofc['timestamp'].dt.year == 2023]
sgp_dispatch_ofc.set_index('timestamp',inplace=True)

resample_sgp_dispatch_hourly = sgp_dispatch_ofc.resample('H').median(numeric_only=True)
resample_sgp_dispatch = resample_sgp_dispatch_hourly.resample('D').sum()

# plot daily dispatch
fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=(resample_sgp_dispatch['demand_mw'] - resample_sgp_dispatch['solar_mw']).div(1e3), 
        mode='lines',
        name='gas',
        line=dict(color='black'),
        stackgroup='one'
        )
)
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=(resample_sgp_dispatch['solar_mw']).div(1e3), 
        mode='lines',
        name='solar',
        line=dict(color='yellow'),
        stackgroup='one'
        )
)
fig.update_layout(
        xaxis=dict(
            rangeslider=dict(
                visible=True
            ),
            type="date",
        ),
        # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
        yaxis_title=f'Generation (GW)',
        xaxis_title='Time (Day)',
        title='SGP daily dispatch 2023 (ofc)',
        width=1000,
        height=500,
    )

fig.show()

In [ ]:
# --- CALIBRATION 3c: DISPATCH MONTHLY 2023 - Malaysia (model run) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='MYS',resample='D', show_exports=True, show_imports=True)
fig.update_layout(title_text='MYS dispatch 2023 (model run)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3d: DISPATCH MONTHLY 2023 - Malaysia (ofc) --- #
# Note: this is the monthly averaged dispatch and demand profile for 2023

mys_dispatch_ofc = pd.read_csv('/home/yantidiah/tza-pypsa/bq_export_malaysia_gso.csv')
mys_dispatch_ofc['DT'] = pd.to_datetime(mys_dispatch_ofc['DT'])
mys_dispatch_ofc = mys_dispatch_ofc[mys_dispatch_ofc['DT'].dt.year == 2023]
mys_dispatch_ofc.set_index('DT',inplace=True)
mys_dispatch_ofc = mys_dispatch_ofc.drop(mys_dispatch_ofc.columns[-1],axis=1)

resample_mys_dispatch_hourly = mys_dispatch_ofc.resample('H').median()
resample_mys_dispatch = resample_mys_dispatch_hourly.resample('D').sum()

# plot daily dispatch
fig = go.Figure()

for tech in resample_mys_dispatch.columns:
    fig.add_trace(
        go.Scatter(
            x=resample_mys_dispatch.index, 
            y=resample_mys_dispatch[tech].div(1e3), 
            mode='lines',
            name=tech,
            stackgroup='one'
            )
    )

fig.update_layout(
        xaxis=dict(
            rangeslider=dict(
                visible=True
            ),
            type="date",
        ),
        # yaxis_range=(0, (df.drop('timestep',axis=1).sum(axis=1).max() / 1e3)*1.05 ),
        yaxis_title=f'Generation (GW)',
        xaxis_title='Time (Day)',
        title='MYS daily dispatch 2023 (ofc)',
        width=1000,
        height=500,
    )

fig.show()

In [ ]:
# --- CALIBRATION 3e: DISPATCH MONTHLY 2023 - Singapore (ofc total vs model per type) --- #

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='SGP',resample='D', show_exports=True, show_imports=True)
fig.add_trace(
    go.Scatter(
        x=resample_sgp_dispatch.index, 
        y=resample_sgp_dispatch['demand_mw'].div(1e3), 
        mode='lines',
        name='demand_official',
        line=dict(color='red'),
        # stackgroup='one'
        )
)
fig.update_layout(title_text='SGP dispatch 2023 (model run vs total dispatched demand)',width=1000,height=500)
fig.show()

In [ ]:
# --- CALIBRATION 3f: DISPATCH MONTHLY 2023 - Malaysia (ofc total vs model per type) --- #

fig = tz_pypsa.plotting.dispatch(network,'2023',iso_code='MYS',resample='D', show_exports=True, show_imports=True)
fig.add_trace(
    go.Scatter(
        x=resample_mys_dispatch.index, 
        y=((resample_mys_dispatch['Coal']+
           resample_mys_dispatch['Gas']+
           resample_mys_dispatch['CoGen']+
           resample_mys_dispatch['Oil']+
           resample_mys_dispatch['Hydro']+
           resample_mys_dispatch['Solar'])
           .div(1e3)), 
        mode='lines',
        name='demand_official',
        line=dict(color='red'),
        # stackgroup='one'
        )
)
fig.update_layout(title_text='MYS dispatch 2023 (model run vs total dispatched generation)',width=1000,height=500)
fig.show()